In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check the GPU status
!nvidia-smi

Fri Sep 25 15:53:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [4]:
import os
from datetime import datetime
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from peft import PeftModel
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file

# Configurations

In [5]:
# Run configuration
SRC_LANG = 'en'
TGT_LANG = 'el'

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
LORA_LANG_ID = 'alxxtexxr/XLM-R-Base-wikipedia-el-4K-s42-LoRA-v260925004156'
LORA_LANG_CKPT_STEP = 900
LORA_TASK_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LoRA-v260925013329'
LORA_TASK_CKPT_STEP = 3300
ADDITION_TYPE = 'avg'

# Set up the addition weights
addition_weights = [0.5, 0.5] if ADDITION_TYPE == 'avg' else [1.0, 1.0]

# Set up the hub merged model ID
assert SRC_LANG in LORA_TASK_ID and TGT_LANG in LORA_LANG_ID, "LoRA IDs do not match the specified source language and target language."
model_id_pt_0, model_id_pt_1 = LORA_TASK_ID.split(SRC_LANG)
model_id_pt_1_0 = model_id_pt_1.split("K-")[-1].split("-v")[0]
hub_merged_model_id = f"{model_id_pt_0}{TGT_LANG}-{model_id_pt_1_0}-{ADDITION_TYPE}-v{datetime.now().strftime("%y%m%d%H%M%S")}"
print(f"Hub merged model ID: {hub_merged_model_id}")

Hub merged model ID: alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333


# Utilities

In [6]:
# LoRA utilities
def download_hf_model(
        repo_id, 
        ckpt_step, 
        max_checkpoints=10_000,
        ckpt_interval=25,
    ):
    local_dir = repo_id.split('/')[-1]
    ignore_checkpoints = None
    
    if ckpt_step is not None:
        ignore_checkpoints = [f'checkpoint-{i}/*' for i in range(0, max_checkpoints, ckpt_interval) if i != ckpt_step]

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        ignore_patterns=ignore_checkpoints,
    )

    ckpt_dir = None
    if ckpt_step is not None:
        ckpt_dir = os.path.join(local_dir, f'checkpoint-{ckpt_step}')
    return local_dir, ckpt_dir

# Model

In [7]:
# Download the language and task LoRA adapters
_, lora_lang_dir = download_hf_model(repo_id=LORA_LANG_ID, ckpt_step=LORA_LANG_CKPT_STEP)
_, lora_task_dir = download_hf_model(repo_id=LORA_TASK_ID, ckpt_step=LORA_TASK_CKPT_STEP)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

In [8]:
# Create directories for fixed LoRA adapters
lora_lang_fixed_dir = f'{lora_lang_dir}_fixed'
lora_task_fixed_dir = f'{lora_task_dir}_fixed'

!mkdir -p $lora_lang_fixed_dir
!mkdir -p $lora_task_fixed_dir
!cp -r $lora_lang_dir/* $lora_lang_fixed_dir
!cp -r $lora_task_dir/* $lora_task_fixed_dir
!rm $lora_lang_fixed_dir/adapter_model.safetensors
!rm $lora_task_fixed_dir/adapter_model.safetensors

In [9]:
# Load the LoRA state dicts and fix them
lora_lang_state_dict = load_file(f'{lora_lang_dir}/adapter_model.safetensors')
lora_task_state_dict = load_file(f'{lora_task_dir}/adapter_model.safetensors')

lora_lang_fixed_state_dict = {k: v for k, v in lora_lang_state_dict.items() if 'lm_head' not in k}
lora_task_fixed_state_dict = lora_task_state_dict

save_file(lora_lang_fixed_state_dict, f'{lora_lang_fixed_dir}/adapter_model.safetensors')
save_file(lora_task_fixed_state_dict, f'{lora_task_fixed_dir}/adapter_model.safetensors')

In [10]:
# Sanity check
for i in range(11):
    print(f"Task LoRA layer-{i} attention value weight norm:", lora_task_fixed_state_dict[f'base_model.model.roberta.encoder.layer.{i}.attention.self.value.lora_A.weight'].norm().item())
print()
print("Task LoRA qa_outputs weight norm:", lora_task_fixed_state_dict['base_model.model.qa_outputs.weight'].norm().item())
print("Task LoRA qa_outputs bias norm:", lora_task_fixed_state_dict['base_model.model.qa_outputs.bias'].norm().item())

Task LoRA layer-0 attention value weight norm: 1.8784334659576416
Task LoRA layer-1 attention value weight norm: 2.0121586322784424
Task LoRA layer-2 attention value weight norm: 2.0960187911987305
Task LoRA layer-3 attention value weight norm: 2.291851043701172
Task LoRA layer-4 attention value weight norm: 2.1963884830474854
Task LoRA layer-5 attention value weight norm: 2.2859787940979004
Task LoRA layer-6 attention value weight norm: 2.2279810905456543
Task LoRA layer-7 attention value weight norm: 2.1960861682891846
Task LoRA layer-8 attention value weight norm: 2.249143123626709
Task LoRA layer-9 attention value weight norm: 2.292754650115967
Task LoRA layer-10 attention value weight norm: 1.9429188966751099

Task LoRA qa_outputs weight norm: 1.178995966911316
Task LoRA qa_outputs bias norm: 0.008219932205975056


In [11]:
from peft import PeftConfig

# A trick because peft makes the head of a question-answering model to be two layers: the original and trainable qa_outputs layers
# This seems to make the LoRA merging to not work as expected internally and make the LoRA-merged model to perform poorly
config = PeftConfig.from_pretrained(lora_task_fixed_dir)
config.task_type = 'FEATURE_EXTRACTION'
config.modules_to_save = None
config.save_pretrained(lora_task_fixed_dir)

/usr/local/lib/python3.13/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'kasa_config', 'lora_ga_config', 'monteclora_config', 'peft_version', 'use_bdlora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


In [12]:
# Load the base model
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID, device_map='auto')

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# Sanity check
for i in range(11):
    print(f"Base model layer-{i} attention value weight norm:", base_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Base model qa_outputs weight norm:", base_model.qa_outputs.weight.norm().item())
print("Base model qa_outputs bias norm:", base_model.qa_outputs.bias.norm().item())

Base model layer-0 attention value weight norm: 28.364133834838867
Base model layer-1 attention value weight norm: 28.845458984375
Base model layer-2 attention value weight norm: 30.10150146484375
Base model layer-3 attention value weight norm: 36.34113693237305
Base model layer-4 attention value weight norm: 38.29389953613281
Base model layer-5 attention value weight norm: 41.62784957885742
Base model layer-6 attention value weight norm: 38.20062255859375
Base model layer-7 attention value weight norm: 38.956634521484375
Base model layer-8 attention value weight norm: 39.158668518066406
Base model layer-9 attention value weight norm: 34.89980697631836
Base model layer-10 attention value weight norm: 31.030277252197266

Base model qa_outputs weight norm: 0.7969438433647156
Base model qa_outputs bias norm: 0.0


In [14]:
# Load the task and language LoRA adapters into the base model
# lora_model = PeftModel.from_pretrained(base_model, model_id=LORA_TASK_ID, subfolder=LORA_TASK_CKPT_DIR, adapter_name='task')
# lora_model.load_adapter(model_id=LORA_LANG_ID, subfolder=LORA_LANG_CKPT_DIR, adapter_name='lang')
lora_model = PeftModel.from_pretrained(base_model, lora_task_fixed_dir, adapter_name='task')
lora_model.load_adapter(lora_lang_fixed_dir, adapter_name='lang')

# Combine the task and language LoRA adapters into a single LoRA adapter via weighted addition
lora_model.add_weighted_adapter(
    adapters=['task', 'lang'],
    weights=addition_weights,
    combination_type='linear',
    adapter_name='lora_addition'
)
lora_model.set_adapter('lora_addition')

# Merge the combined LoRA adapter into the base model
merged_model = lora_model.merge_and_unload()

In [15]:
# Sanity check
for i in range(11):
    print(f"Merged model layer-{i} attention value weight norm:", merged_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Merged model qa_outputs weight norm:", merged_model.qa_outputs.weight.norm().item())
print("Merged model qa_outputs bias norm:", merged_model.qa_outputs.bias.norm().item())

Merged model layer-0 attention value weight norm: 28.36728286743164
Merged model layer-1 attention value weight norm: 28.849205017089844
Merged model layer-2 attention value weight norm: 30.105112075805664
Merged model layer-3 attention value weight norm: 36.34761428833008
Merged model layer-4 attention value weight norm: 38.30080032348633
Merged model layer-5 attention value weight norm: 41.63356399536133
Merged model layer-6 attention value weight norm: 38.20808029174805
Merged model layer-7 attention value weight norm: 38.965782165527344
Merged model layer-8 attention value weight norm: 39.16653823852539
Merged model layer-9 attention value weight norm: 34.90757369995117
Merged model layer-10 attention value weight norm: 31.033695220947266

Merged model qa_outputs weight norm: 1.1789958477020264
Merged model qa_outputs bias norm: 0.008219932205975056


In [ ]:
# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...dtjyihy/model.safetensors:   4%|3         | 39.9MB / 1.11GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp4rynf01m/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

  ...m/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

Merged model uploaded to: https://huggingface.co/alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333


: 